# iProperty 100k scraper — v2 (expanded states + early termination fix)

Patches in this version:
- **Expanded to 16 Malaysian states** (was 4) — the main fix for running out of records
- **True early termination** — once a base URL returns 0 cards, all deeper pages are skipped immediately
- **MAX_PAGE_PER_URL raised to 100** — squeezes more from high-volume states
- All previous fixes retained (junk row filter, dedup, resume logic, etc.)

In [1]:
# Run this once in Colab. If you are on local Jupyter and already installed these, you may skip it.
%pip -q install selenium undetected-chromedriver beautifulsoup4 lxml pandas setuptools

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Optional: check your current CSV size before scraping
import os, pandas as pd
for f in ["iproperty_data.csv", "iproperty_data(2).csv"]:
    if os.path.exists(f):
        df_check = pd.read_csv(f, low_memory=False)
        print(f"{f}: {len(df_check):,} rows")
        if "listing_type" in df_check.columns:
            print(df_check["listing_type"].value_counts(dropna=False))

iproperty_data.csv: 99,841 rows
listing_type
for-sale    64110
for-rent    35731
Name: count, dtype: int64


In [3]:
import sys
import setuptools._distutils.version
sys.modules['distutils.version'] = setuptools._distutils.version

import re
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from bs4 import BeautifulSoup
import pandas as pd
import time
import os
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

# ─────────────────────────────────────────────────────────────────────────────
# DEBUG MODE — flip True for a one-shot HTML dump, then inspect debug_dump.html
# ─────────────────────────────────────────────────────────────────────────────
DEBUG_MODE = False

# ─────────────────────────────────────────────────────────────────────────────
# CSS / DOM SELECTORS
# ─────────────────────────────────────────────────────────────────────────────
SELECTORS = {
    "card":          'article, div[class*="Listing"], div[class*="listing"], div[class*="Card"], div[class*="card"]',
    "card_alt":      'a[href*="/property-for-sale/"], a[href*="/property-for-rent/"], a[href*="/property/"]',

    "title":         'h2, h3, [class*="title"], [class*="Title"]',
    "price":         '[class*="price"], [class*="Price"]',
    "location":      '[class*="location"], [class*="Location"], [class*="address"], [class*="Address"]',
    "bedrooms":      '[class*="bed"], [aria-label*="bed"]',
    "bathrooms":     '[class*="bath"], [aria-label*="bath"]',
    "size":          '[class*="floor"], [class*="size"], [class*="area"], [aria-label*="sqft"]',
    "property_type": '[class*="property-type"], [class*="PropertyType"], [class*="type"]',
    "agent":         '[class*="agent"], [class*="Agent"]',
    "link":          'a[href*="/property-for-sale/"], a[href*="/property-for-rent/"], a[href*="/property/"], a[href*="/for-sale/"], a[href*="/for-rent/"]',

    "wait_for":      'body',
}

# ─────────────────────────────────────────────────────────────────────────────
# URL SOURCES
# FIX 1: Added property-for-rent alongside for-sale — this is the main new
#         source that gets you past the ~91k for-sale ceiling.
# FIX 2: Removed property-type sub-URLs (they duplicate the main state URL
#         and waste scraping time — dedup caught them but slowed everything).
# ─────────────────────────────────────────────────────────────────────────────
LISTING_TYPES = [
    "property-for-sale",
    "property-for-rent",
]

STATES = [
    # High-volume states (already partially scraped)
    "selangor-45nk1",
    "kuala-lumpur-58jok",
    "johor-2hh35",
    "penang-5qvq6",
    # Additional states to fill the gap to 100k
    "sabah-bsqr",
    "sarawak-bsqs",
    "negeri-sembilan-2hh40",
    "perak-2hh3b",
    "pahang-2hh39",
    "kedah-2hh36",
    "melaka-2hh38",
    "terengganu-2hh3d",
    "kelantan-2hh37",
    "perlis-2hh3a",
    "putrajaya-58joj",
    "labuan-bsqp",
]

MAX_PAGE_PER_URL = 100
PROGRESS_FILE = "iproperty_progress_FINAL_500_RENT.txt"
MAX_RECORDS = 100_000
NUM_WORKERS = 3
PAGE_LOAD_WAIT = 6.0

# FIX 3: Reduced MAX_PAGE_PER_URL from 1000 → 60.
# iProperty hard-caps results at ~30–50 pages per search regardless of what
# page number you request. Pages 51-1000 always return 0 cards, so your
# early-termination logic in the worker handles it, but setting this to 60
# avoids generating 148,500 dead URLs that burn time just to be skipped.
MAX_PAGE_PER_URL = 100
LISTINGS_PER_PAGE = 30


def build_urls() -> list[str]:
    urls = []
    base = "https://www.iproperty.com.my"

    for ltype in LISTING_TYPES:
        for state in STATES:
            for page in range(1, MAX_PAGE_PER_URL + 1):
                path = f"/{ltype}/in-{state}"
                url = f"{base}{path}" if page == 1 else f"{base}{path}?page={page}"
                urls.append(url)

    # Prioritize pages that usually still contain real cards
    def page_score(url):
        m = re.search(r'page=(\d+)', url)
        page = int(m.group(1)) if m else 1

        # Broad high-volume states first
        if "selangor" in url:
            state_score = 0
        elif "kuala-lumpur" in url:
            state_score = 1
        elif "johor" in url:
            state_score = 2
        elif "penang" in url:
            state_score = 3
        else:
            state_score = 4

        # Pages 1–30 are usually more useful than deep pages
        return (state_score, page)

    urls = sorted(urls, key=page_score)
    return urls


URLS = build_urls()
print(f"📋 Total candidate URLs: {len(URLS):,}  "
      f"(theoretical max ≈ {len(URLS) * LISTINGS_PER_PAGE:,} records)")

# ─────────────────────────────────────────────────────────────────────────────
# Config
# ─────────────────────────────────────────────────────────────────────────────
# In Colab/Notebook, keep using your existing CSV, but use a FRESH progress file.
# Reason: your old notebook may have marked pages as completed even when it parsed 0 cards.
def _first_existing(*names):
    for name in names:
        if os.path.exists(name):
            return name
    return names[0]

CSV_FILE      = _first_existing("iproperty_data.csv", "iproperty_data(2).csv")
PROGRESS_FILE = "iproperty_progress_FIXED.txt"
MAX_RECORDS   = 100_000    # target for the assignment
TRIM_TO_TARGET = True
SAVE_EVERY    = 100
PAGE_LOAD_WAIT = 4.0
NUM_WORKERS   = 5

# ─────────────────────────────────────────────────────────────────────────────
# Shared state
# ─────────────────────────────────────────────────────────────────────────────
results_lock  = threading.Lock()
seen_ids_lock = threading.Lock()
progress_lock = threading.Lock()
stop_event    = threading.Event()

results        = []
seen_ids       = set()
completed_urls = set()

# ─────────────────────────────────────────────────────────────────────────────
# FIX 4: Resume logic — skip junk agent rows when reloading old CSV so they
# don't pollute seen_ids and inflate the count toward MAX_RECORDS.
# ─────────────────────────────────────────────────────────────────────────────
_JUNK_URL_RE = re.compile(r'/(property-agent|developer|project)/', re.I)

if os.path.exists(CSV_FILE):
    print(f"📂 Resuming from '{CSV_FILE}'…")
    df_old = pd.read_csv(CSV_FILE, low_memory=False)

    # Drop junk rows (agent pages scraped as listings)
    if "listing_id" in df_old.columns:
        junk_mask = df_old["listing_id"].astype(str).str.match(r'https?://') & \
                    df_old["listing_id"].astype(str).str.contains(
                        r'/(property-agent|developer)/', regex=True
                    )
        n_junk = junk_mask.sum()
        if n_junk:
            df_old = df_old[~junk_mask]
            print(f"   ↳ Dropped {n_junk:,} junk agent rows from old CSV.")

        # FIX 5: Convert URL-style listing_ids to numeric sale IDs for clean dedup.
        # Old rows used the full URL as ID; extract the sale-XXXXXX number.
        def extract_sale_id(val):
            s = str(val)
            # Already in new format: "sale-500769308" or "rent-500769308"
            if re.match(r'(?:sale|rent)-\d+', s):
                return s
            # Old format: full URL — extract prefixed form
            m = re.search(r'((?:sale|rent)-\d+)', s)
            if m:
                return m.group(1)
            # Bare numeric ID from previous run — assume for-sale
            if re.match(r'^\d+$', s):
                return f"sale-{s}"
            return s

        df_old["listing_id"] = df_old["listing_id"].apply(extract_sale_id)
        seen_ids = set(df_old["listing_id"].dropna().astype(str))
        results  = df_old.to_dict(orient="records")
        print(f"   ↳ {len(seen_ids):,} real listings already collected.")
    else:
        print("   ⚠️  Old CSV missing 'listing_id' — duplicates may occur.")
else:
    print("🚀 Starting fresh scrape.")

if os.path.exists(PROGRESS_FILE):
    with open(PROGRESS_FILE) as f:
        completed_urls = {line.strip() for line in f if line.strip()}
    print(f"   ↳ {len(completed_urls):,} URLs already done — skipping.\n")

_last_save_count = len(results)


# ─────────────────────────────────────────────────────────────────────────────
# Persistence
# ─────────────────────────────────────────────────────────────────────────────
def save():
    global _last_save_count
    with results_lock:
        df_out = pd.DataFrame(results)
        if TRIM_TO_TARGET and len(df_out) > MAX_RECORDS:
            df_out = df_out.head(MAX_RECORDS)
        df_out.to_csv(CSV_FILE, index=False)
        _last_save_count = len(results)
        print(f"   💾 [{threading.current_thread().name}] "
              f"Saved {len(df_out):,} records → {CSV_FILE}")


def mark_url_done(url: str):
    with progress_lock:
        with open(PROGRESS_FILE, "a") as f:
            f.write(url + "\n")
        completed_urls.add(url)


# ─────────────────────────────────────────────────────────────────────────────
# Browser factory
# ─────────────────────────────────────────────────────────────────────────────
_driver_init_lock = threading.Lock()


# Set this from your error message:
# Current browser version is 148.0.7778.179  → use 148
CHROME_MAJOR_VERSION = 148
_uc_cache_cleared = False


def _clear_uc_cache_once():
    """Remove cached undetected_chromedriver binaries so v149 does not get reused."""
    global _uc_cache_cleared
    if _uc_cache_cleared:
        return
    _uc_cache_cleared = True

    import shutil
    from pathlib import Path

    possible_dirs = [
        Path.home() / "AppData" / "Roaming" / "undetected_chromedriver",
        Path.home() / "AppData" / "Local" / "undetected_chromedriver",
        Path.home() / ".local" / "share" / "undetected_chromedriver",
        Path.home() / ".cache" / "undetected_chromedriver",
    ]

    for d in possible_dirs:
        if d.exists():
            try:
                shutil.rmtree(d)
                print(f"🧹 Cleared old ChromeDriver cache: {d}")
            except Exception as e:
                print(f"⚠️ Could not clear cache {d}: {e}")


def make_driver() -> uc.Chrome:
    with _driver_init_lock:
        opts = uc.ChromeOptions()
        opts.add_argument("--blink-settings=imagesEnabled=false")
        opts.add_argument("--no-sandbox")
        opts.add_argument("--disable-dev-shm-usage")
        opts.add_argument("--disable-extensions")
        opts.add_argument("--disable-gpu")
        opts.add_argument("--window-size=1920,1080")
        opts.add_argument(
            "--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/125.0.0.0 Safari/537.36"
        )
        # IMPORTANT: force ChromeDriver to match your installed Chrome.
        # Your error says browser = 148, but driver = 149, so force version_main=148.
        _clear_uc_cache_once()
        print(f"🔧 Launching ChromeDriver for Chrome major version {CHROME_MAJOR_VERSION}...")
        return uc.Chrome(options=opts, version_main=CHROME_MAJOR_VERSION)


# ─────────────────────────────────────────────────────────────────────────────
# Selector helpers
# ─────────────────────────────────────────────────────────────────────────────
def select_one_text(element, selector: str) -> str | None:
    for sel in [s.strip() for s in selector.split(",")]:
        try:
            el = element.select_one(sel)
            if el:
                return el.get_text(strip=True)
        except Exception:
            pass
    return None


def select_one_attr(element, selector: str, attr: str) -> str | None:
    for sel in [s.strip() for s in selector.split(",")]:
        try:
            el = element.select_one(sel)
            if el and el.get(attr):
                return el[attr].strip()
        except Exception:
            pass
    return None


# ─────────────────────────────────────────────────────────────────────────────
# Card parser
# FIX 6: listing_id is now always the numeric sale-XXXXXXX number.
#         Agent/developer pages are explicitly skipped before any processing.
# ─────────────────────────────────────────────────────────────────────────────
def parse_cards(page_source: str, current_url: str, region: str,
                listing_type: str) -> tuple[list[dict], list[str]]:
    soup  = BeautifulSoup(page_source, "lxml")
    cards = soup.select(SELECTORS["card"])
    if not cards:
        cards = soup.select(SELECTORS["card_alt"])
    if not cards:
        cards = soup.select(
            'a[href*="/property-for-sale/"], '
            'a[href*="/property-for-rent/"], '
            'a[href*="/property-"]'
        )

    with seen_ids_lock:
        local_seen = frozenset(seen_ids)

    new_records: list[dict] = []
    new_ids:     list[str]  = []

    for card in cards:
        # ── URL ────────────────────────────────────────────────────────────
        link_el  = card.select_one(SELECTORS["link"]) or card.find("a", href=True)
        raw_href = link_el["href"] if link_el and link_el.get("href") else None

        if not raw_href:
            continue

        listing_url = (
            raw_href if raw_href.startswith("http")
            else "https://www.iproperty.com.my" + raw_href.split("?")[0]
        )

        # FIX 6a: Skip agent / developer / non-property pages entirely
        if _JUNK_URL_RE.search(listing_url):
            continue

        # FIX 6b: Extract ID from sale-XXXXXXX or rent-XXXXXXX.
        # Keep the prefix so sale-12345 and rent-12345 are distinct entries.
        m = re.search(r'((?:sale|rent)-\d+)', listing_url)
        if not m:
            continue
        listing_id = m.group(1)   # e.g. "sale-500769308" or "rent-500769308"

        if listing_id in local_seen or listing_id in new_ids:
            continue

        # ── Core fields ────────────────────────────────────────────────────
        title         = select_one_text(card, SELECTORS["title"])
        price         = select_one_text(card, SELECTORS["price"])
        location      = select_one_text(card, SELECTORS["location"])
        bedrooms      = select_one_text(card, SELECTORS["bedrooms"])
        bathrooms     = select_one_text(card, SELECTORS["bathrooms"])
        size          = select_one_text(card, SELECTORS["size"])
        property_type = select_one_text(card, SELECTORS["property_type"])
        agent         = select_one_text(card, SELECTORS["agent"])

        if not title and not price:
            continue

        # ── Extras from raw text ───────────────────────────────────────────
        card_text = card.get_text(" ", strip=True)

        tenure_m   = re.search(r'\b(Freehold|Leasehold|Malay Reserve|SOHO|SOVO)\b',
                                card_text, re.I)
        furnish_m  = re.search(r'\b(Fully Furnished|Partially Furnished|Unfurnished)\b',
                                card_text, re.I)
        carpark_m  = re.search(r'(\d+)\s*(?:car\s*park|parking|carpark)', card_text, re.I)

        new_ids.append(listing_id)
        new_records.append({
            "listing_id":    listing_id,
            "listing_type":  listing_type,   # "for-sale" or "for-rent"
            "title":         title,
            "price":         price,
            "location":      location,
            "bedrooms":      bedrooms,
            "bathrooms":     bathrooms,
            "size_sqft":     size,
            "property_type": property_type,
            "tenure":        tenure_m.group(1).title() if tenure_m else None,
            "furnishing":    furnish_m.group(1).title() if furnish_m else None,
            "car_parks":     carpark_m.group(1) if carpark_m else None,
            "agent":         agent,
            "listing_url":   listing_url,
            "source_url":    current_url,
            "region":        region,
        })

    return new_records, new_ids


# ─────────────────────────────────────────────────────────────────────────────
# Region / listing-type helpers
# ─────────────────────────────────────────────────────────────────────────────
_SLUG_RE = re.compile(r'-[a-z0-9]+$')


def region_from_url(url: str) -> str:
    try:
        part = re.split(r'/in-', url)[1].split("?")[0].split("/")[0]
        return _SLUG_RE.sub("", part).replace("-", " ").title()
    except Exception:
        return "Unknown"


def listing_type_from_url(url: str) -> str:
    if "property-for-rent" in url:
        return "for-rent"
    return "for-sale"


# ─────────────────────────────────────────────────────────────────────────────
# Per-URL scraper
# ─────────────────────────────────────────────────────────────────────────────
def scrape_url(driver, url: str, worker_name: str) -> int:
    global _last_save_count
    print(f"\n[{worker_name}] 🌐 {url}")

    try:
        driver.get(url)
    except Exception as e:
        print(f"[{worker_name}] ❌ Navigation error: {e}")
        return 0

    try:
        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, SELECTORS["wait_for"]))
        )
    except TimeoutException:
        print(f"[{worker_name}] ⏱ Timed out waiting for page.")
        return 0

    time.sleep(PAGE_LOAD_WAIT)

    if DEBUG_MODE:
        dump = r"c:\SEM6\HPDP\iproperty_scrape\debug_dump.html"
        with open(dump, "w", encoding="utf-8") as f:
            f.write(driver.page_source)
        print(f"\n[DEBUG] HTML dumped → {dump}")
        stop_event.set()
        return 0

    region       = region_from_url(url)
    listing_type = listing_type_from_url(url)

    driver.execute_script("window.scrollTo(0, document.body.scrollHeight * 0.5);")
    time.sleep(0.8)
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(0.8)

    new_records, new_ids = parse_cards(
        driver.page_source, url, region, listing_type
    )

    if not new_records:
        print(f"[{worker_name}] ℹ️  0 new cards parsed — skipping.")
        return 0

    with seen_ids_lock:
        truly_new = [
            (rec, lid)
            for rec, lid in zip(new_records, new_ids)
            if lid not in seen_ids
        ]
        for _, lid in truly_new:
            seen_ids.add(lid)

    if not truly_new:
        print(f"[{worker_name}] ℹ️  All cards already seen — skipping.")
        return 0

    records_only = [r for r, _ in truly_new]
    with results_lock:
        results.extend(records_only)
        current_total = len(results)
        should_save   = (current_total - _last_save_count) >= SAVE_EVERY

    print(f"[{worker_name}] 📊 +{len(records_only)} new  "
          f"(total: {current_total:,} / {MAX_RECORDS:,})")

    if should_save:
        save()

    if current_total >= MAX_RECORDS:
        stop_event.set()
        print(f"[{worker_name}] 🏁 MAX_RECORDS ({MAX_RECORDS:,}) reached — stopping.")

    return len(records_only)


# ─────────────────────────────────────────────────────────────────────────────
# Worker
# ─────────────────────────────────────────────────────────────────────────────
def worker(url_queue: list[str], worker_id: int):
    name = f"W{worker_id}"
    print(f"[{name}] 🚀 Launching browser…")
    driver = make_driver()
    print(f"[{name}] ✅ Browser ready. {len(url_queue):,} URLs assigned.")

    # Track base URLs that returned 0 cards — skip deeper pages immediately
    exhausted_bases: set[str] = set()

    try:
        for url in url_queue:
            if stop_event.is_set():
                break

            with progress_lock:
                if url in completed_urls:
                    continue

            # FIX 7: Skip deeper pages once a base URL is exhausted
            base_url = re.sub(r'[?&]page=\\d+', '', url).rstrip('&?')
            if base_url in exhausted_bases:
                mark_url_done(url)  # mark as done so it won't be retried
                continue

            added = scrape_url(driver, url, name)
            mark_url_done(url)

            if added == 0:
                exhausted_bases.add(base_url)
                print(f"[{name}] ⏭  Base URL exhausted — skipping deeper pages: {base_url}")

            time.sleep(2.5)

    finally:
        driver.quit()
        print(f"[{name}] 🛑 Browser closed.")


# ─────────────────────────────────────────────────────────────────────────────
# URL distribution
# ─────────────────────────────────────────────────────────────────────────────
def distribute(urls: list[str], n: int) -> list[list[str]]:
    buckets = [[] for _ in range(n)]
    for i, url in enumerate(urls):
        buckets[i % n].append(url)
    return buckets


# ─────────────────────────────────────────────────────────────────────────────
# Main
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    urls_to_scrape = [u for u in URLS if u not in completed_urls]
    print(f"\n📋 {len(urls_to_scrape):,} URLs to scrape  "
          f"({len(completed_urls):,} already done)  "
          f"across {NUM_WORKERS} workers.\n")

    if not urls_to_scrape:
        print("Nothing to do — all URLs already completed.")
        sys.exit(0)

    print("🔧 Pre-checking ChromeDriver…")
    try:
        temp = make_driver()
        temp.quit()
        print("✅ ChromeDriver OK.\n")
    except Exception as e:
        print(f"❌ ChromeDriver failed: {e}")
        sys.exit(1)

    n_workers   = min(NUM_WORKERS, len(urls_to_scrape))
    url_buckets = distribute(urls_to_scrape, n_workers)

    with ThreadPoolExecutor(max_workers=n_workers,
                            thread_name_prefix="Scraper") as pool:
        futures = {
            pool.submit(worker, bucket, wid): wid
            for wid, bucket in enumerate(url_buckets, 1)
        }
        for future in as_completed(futures):
            wid = futures[future]
            try:
                future.result()
            except Exception as exc:
                print(f"[W{wid}] ⚠️  Worker crashed: {exc}")

    save()
    final_rows = len(pd.read_csv(CSV_FILE, low_memory=False)) if os.path.exists(CSV_FILE) else 0
    print(f"\n🏁 Done. {final_rows:,} records saved to '{CSV_FILE}'.")
    if final_rows >= 100_000:
        print("✅ Target reached: 100,000 records.")
    else:
        print(f"⚠️ Still short by {100_000 - final_rows:,} records. Re-run this cell to continue.")

📋 Total candidate URLs: 3,200  (theoretical max ≈ 96,000 records)
📂 Resuming from 'iproperty_data.csv'…


C:\Users\piyat\AppData\Local\Temp\ipykernel_61380\2462957645.py:174: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df_old["listing_id"].astype(str).str.contains(


   ↳ 99,841 real listings already collected.
   ↳ 314 URLs already done — skipping.


📋 2,886 URLs to scrape  (314 already done)  across 5 workers.

🔧 Pre-checking ChromeDriver…
🧹 Cleared old ChromeDriver cache: C:\Users\piyat\AppData\Roaming\undetected_chromedriver
🔧 Launching ChromeDriver for Chrome major version 148...
✅ ChromeDriver OK.

[W1] 🚀 Launching browser…
🔧 Launching ChromeDriver for Chrome major version 148...
[W2] 🚀 Launching browser…
[W3] 🚀 Launching browser…
[W4] 🚀 Launching browser…
[W5] 🚀 Launching browser…
[W1] ✅ Browser ready. 578 URLs assigned.🔧 Launching ChromeDriver for Chrome major version 148...


[W1] 🌐 https://www.iproperty.com.my/property-for-sale/in-selangor-45nk1?page=81
[W2] ✅ Browser ready. 577 URLs assigned.🔧 Launching ChromeDriver for Chrome major version 148...


[W2] 🌐 https://www.iproperty.com.my/property-for-rent/in-selangor-45nk1?page=81
[W3] ✅ Browser ready. 577 URLs assigned.🔧 Launching ChromeDriver for Chrome major version 148...


[W3] 🌐 https